In [0]:
# 1. Get the list of unique years from your SQL table as a list of strings
year_data = spark.sql("SELECT DISTINCT YEAR(order_in_date) as yr FROM my_project_db.gold_sales_fact ORDER BY yr DESC").collect()
years = [str(row['yr']) for row in year_data]

# 2. Create the dropdown using the dynamic list
dbutils.widgets.dropdown("selected_year", years[0], years)

# 3. Retrieve the selected value for use in subsequent SQL cells
selected_year = dbutils.widgets.get("selected_year")

In [0]:
%sql
WITH RegionalTotals AS (
    SELECT 
        l.region,
        ROUND(SUM(f.order_value), 2) AS total_region_revenue
    FROM my_project_db.gold_sales_fact f
    JOIN my_project_db.gold_dim_leads l ON f.lead_id = l.lead_id
    WHERE YEAR(f.order_in_date) = :selected_year
    GROUP BY l.region
    ORDER BY total_region_revenue DESC
    LIMIT 5
),
DivisionPerformance AS (
    SELECT 
        l.region,
        p.division,
        ROUND(SUM(f.order_value), 2) AS division_revenue,
        ROW_NUMBER() OVER(PARTITION BY l.region ORDER BY SUM(f.order_value) DESC) as div_rank
    FROM my_project_db.gold_sales_fact f
    JOIN my_project_db.gold_dim_leads l ON f.lead_id = l.lead_id
    JOIN my_project_db.gold_dim_products p ON f.invt_id = p.invt_id
    WHERE YEAR(f.order_in_date) = :selected_year
      AND l.region IN (SELECT region FROM RegionalTotals)
    GROUP BY l.region, p.division
)
SELECT 
    rt.region,
    rt.total_region_revenue,
    dp.division AS top_division,
    dp.division_revenue
FROM RegionalTotals rt
JOIN DivisionPerformance dp ON rt.region = dp.region
WHERE dp.div_rank = 1
ORDER BY rt.total_region_revenue DESC;